# Proyecto — Análisis de fútbol europeo

## Objetivo

Construir una base de datos de fútbol combinando:

1. Clasificaciones de las cinco grandes ligas europeas para las temporadas 2023-24 y 2024-25.
2. Información de jugadores procedente de `european_football_players.csv`.
3. Información de lesiones procedente de `injuries24_25.csv`.


## Estructura del proyecto

**Fase 1 — Scraping:** obtener las clasificaciones de 10 competiciones.

**Fase 2 — Jugadores:** seleccionar las variables necesarias del CSV.

**Fase 3 — Lesiones:** filtrar 24/25 y acumular días/partidos perdidos.

**Fase 4 — Normalización:** hacer coincidir nombres de competiciones y clubes.

**Fase 5 — Integración:** unir jugadores y lesiones.

**Fase 6 — SQL:** exportar/trasladar los datos para consultas y análisis posteriores.

## 1. Imports y funciones

In [442]:
import requests
import json
import pandas as pd
from bs4 import BeautifulSoup

In [443]:
def obtener_soup(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, "html.parser")

    return soup

In [444]:
def extraer_standings(soup):
    for script in soup.find_all("script"):
        texto = script.get_text()

        if "window.standingsData" in texto:
            datos_json = texto.split("window.standingsData", 1)[1]
            datos_json = datos_json.split("=", 1)[1]
            datos_json = datos_json.split(";", 1)[0].strip()

            return json.loads(datos_json)

    return []

In [445]:
def convertir_standings(liga, standings):
    datos = []

    for equipo in standings:
        datos.append([
            liga,
            equipo["team"],
            equipo["pts"],
            equipo["gf"],
            equipo["ga"]
        ])

    return datos

In [446]:
def crear_dataframe_ligas(ligas):
    datos = []

    for liga, url_liga in ligas.items():
        soup_liga = obtener_soup(url_liga)
        standings = extraer_standings(soup_liga)

        datos.extend(convertir_standings(liga, standings))

    return pd.DataFrame(
        datos,
        columns=["league", "club", "points", "goals for", "goals against"]
    )

## 2. URLs de las cinco grandes ligas — dos temporadas

In [447]:
ligas = {
    "Premier League 2024-2025": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Premier-League-2024-2025.php",
    "Ligue 1 2024-2025": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Ligue-1-2024-2025.php",
    "La Liga 2024-2025": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/La-Liga-2024-2025.php",
    "Bundesliga 2024-2025": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Bundesliga-2024-2025.php",
    "Serie A 2024-2025": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Serie-A-2024-2025.php",
    "Premier League 2023-2024": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Premier-League-2023-2024.php",
    "Ligue 1 2023-2024": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Ligue-1-2023-2024.php",
    "La Liga 2023-2024": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/La-Liga-2023-2024.php",
    "Bundesliga 2023-2024": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Bundesliga-2023-2024.php",
    "Serie A 2023-2024": "https://www.sport-histoire.fr/es/Deporte/Futbol/Campeonatos/Serie-A-2023-2024.php"
}

In [448]:
df_ligas = crear_dataframe_ligas(ligas)

df_ligas


,league,club,points,goals for,goals against
0,Premier League 2024-2025,Liverpool FC,84,86,41
1,Premier League 2024-2025,Arsenal FC,74,69,34
2,Premier League 2024-2025,Manchester City FC,71,72,44
3,Premier League 2024-2025,Chelsea FC,69,64,43
4,Premier League 2024-2025,Newcastle United FC,66,68,47
...,...,...,...,...,...
187,Serie A 2023-2024,Empoli FC,36,29,54
188,Serie A 2023-2024,Cagliari Calcio,36,42,68
189,Serie A 2023-2024,Frosinone Calcio,35,44,69
190,Serie A 2023-2024,US Sassuolo Calcio,30,43,75


## 3. Normalización de nombres de liga

En tu notebook original ya habías establecido estas equivalencias. Se mantienen.

In [449]:
df_ligas["league"] = df_ligas["league"].replace({
    "Premier League 2024-2025": "Premier League ENG 2024-25",
    "Ligue 1 2024-2025": "France Ligue 1 FRA 2024-25",
    "La Liga 2024-2025": "Primera División ESP 2024-25",
    "Bundesliga 2024-2025": "Bundesliga 1 ALE 2024-25",
    "Serie A 2024-2025": "Serie A ITA 2024-25",
    "Premier League 2023-2024": "Premier League ENG 2023-24",
    "Ligue 1 2023-2024": "France Ligue 1 FRA 2023-24",
    "La Liga 2023-2024": "Primera División ESP 2023-24",
    "Bundesliga 2023-2024": "Bundesliga 1 ALE 2023-24",
    "Serie A 2023-2024": "Serie A ITA 2023-24"
})

df_ligas.head()


,league,club,points,goals for,goals against
0,Premier League ENG 2024-25,Liverpool FC,84,86,41
1,Premier League ENG 2024-25,Arsenal FC,74,69,34
2,Premier League ENG 2024-25,Manchester City FC,71,72,44
3,Premier League ENG 2024-25,Chelsea FC,69,64,43
4,Premier League ENG 2024-25,Newcastle United FC,66,68,47


## 4. Cargar los CSV de jugadores y lesiones

In [450]:
df_players = pd.read_csv("european_football_players.csv")
df_injuries = pd.read_csv("injuries24_25.csv")

print(df_players.shape)
print(df_injuries.shape)


(2854, 27)
(15603, 11)


## 5. Seleccionar las variables necesarias de jugadores

In [451]:
columnas_players = [
    "Player",
    "Pos",
    "Squad",
    "Comp",
    "MP",
    "Starts",
    "Min",
]

df_players = df_players[columnas_players].copy()

df_players = df_players.rename(columns={
    "Player": "player",
    "Pos": "position",
    "Squad": "club",
    "Comp": "league",
    "MP": "Matches Played",
    "Starts": "Starts",
    "Min": "Minutes"
})

df_players

,player,position,club,league,Matches Played,Starts,Minutes
0,Max Aarons,DF,Bournemouth,eng Premier League,3,1,86
1,Max Aarons,DF,Valencia,es La Liga,4,1,120
2,Rodrigo Abajas,DF,Valencia,es La Liga,1,1,65
3,James Abankwah,DF,Udinese,it Serie A,6,0,88
4,Keyliane Abdallah,FW,Marseille,fr Ligue 1,1,0,3
...,...,...,...,...,...,...,...
2849,Martin Ødegaard,MF,Arsenal,eng Premier League,30,26,2325
2850,Leo Østigård,DF,Hoffenheim,de Bundesliga,11,11,927
2851,Leo Østigård,DF,Rennes,fr Ligue 1,16,14,1227
2852,Milan Đurić,FW,Monza,it Serie A,18,13,1270


In [452]:
df_injuries

,Season,Injury,Days,Games missed,injury_from_parsed,injury_until_parsed,player_name,player_age,player_position,club,league
0,20/21,Syndesmosis ligament tear,43 days,9,1/28/2021,3/11/2021,Alexander Nübel,24,Goalkeeper,Bayern Munich,Bundesliga
1,20/21,Knee injury,37 days,6,3/7/2021,4/12/2021,Ron-Thorben Hoffmann,22,Goalkeeper,Bayern Munich,Bundesliga
2,20/21,Corona virus,21 days,4,2/17/2021,3/9/2021,Benjamin Pavard,25,Centre-Back,Bayern Munich,Bundesliga
3,20/21,bruise,8 days,2,11/6/2020,11/13/2020,Benjamin Pavard,25,Centre-Back,Bayern Munich,Bundesliga
4,20/21,Ligament injury,22 days,2,7/26/2020,8/16/2020,Benjamin Pavard,25,Centre-Back,Bayern Munich,Bundesliga
...,...,...,...,...,...,...,...,...,...,...,...
15598,24/25,Calf muscle tear,22 days,2,3/14/2025,4/4/2025,Taty Castellanos,27,Forward,Lazio,Serie A
15599,24/25,Adductor injury,25 days,6,2/15/2025,3/11/2025,Taty Castellanos,27,Forward,Lazio,Serie A
15600,24/25,Adductor injury,9 days,2,9/16/2024,9/24/2024,Taty Castellanos,27,Forward,Lazio,Serie A
15601,24/25,Ankle injury,6 days,1,2/9/2025,2/14/2025,Boulaye Dia,28,Forward,Lazio,Serie A


## 6. Preparar el DataFrame de lesiones

In [453]:
df_injuries = df_injuries[df_injuries["Season"].isin(["24/25", "23/24"])].copy()

columnas_injuries = [
    "Season",
    "Injury",
    "Days",
    "Games missed",
    "player_name",
    "player_position",
    "club",
    "league"
]

df_injuries = df_injuries[columnas_injuries].copy()

df_injuries = df_injuries.rename(columns={
    "Season": "season",
    "Injury": "injury",
    "Days": "days",
    "Games missed": "games_missed",
    "player_name": "player",
    "player_position": "position",
    "club": "club",
    "league": "league"
})

df_injuries


,season,injury,days,games_missed,player,position,club,league
2507,23/24,muscular problems,3 days,1,Matej Kovar,Goalkeeper,Bayer Leverkusen,Bundesliga
2508,23/24,Hand injury,15 days,4,Edmond Tapsoba,Centre-Back,Bayer Leverkusen,Bundesliga
2509,23/24,Rest,2 days,1,Alejandro Grimaldo,Left-Back,Bayer Leverkusen,Bundesliga
2510,23/24,Ill,3 days,1,Josip Stanisic,Right-Back,Bayer Leverkusen,Bundesliga
2511,23/24,Muscle injury,213 days,37,Arthur,Right-Back,Bayer Leverkusen,Bundesliga
...,...,...,...,...,...,...,...,...
15598,24/25,Calf muscle tear,22 days,2,Taty Castellanos,Forward,Lazio,Serie A
15599,24/25,Adductor injury,25 days,6,Taty Castellanos,Forward,Lazio,Serie A
15600,24/25,Adductor injury,9 days,2,Taty Castellanos,Forward,Lazio,Serie A
15601,24/25,Ankle injury,6 days,1,Boulaye Dia,Forward,Lazio,Serie A


## 7. Normalizar nombres de competiciones

In [454]:
df_players["league"] = df_players["league"].replace({
    "eng Premier League": "Premier League",
    "es La Liga": "La Liga",
    "it Serie A": "Serie A",
    "fr Ligue 1": "Ligue 1",
    "de Bundesliga": "Bundesliga"
})

df_players["league"].unique()


<ArrowStringArray>
['Premier League', 'La Liga', 'Serie A', 'Ligue 1', 'Bundesliga']
Length: 5, dtype: str

## 8. Normalizar nombres de clubes

In [455]:
clubes = {
    "Saint-Étienne": "Saint-Etienne",
    "Alavés": "Deportivo Alaves",
    "Milan": "AC Milan",
    "Freiburg": "SC Freiburg",
    "Dortmund": "Borussia Dortmund",
    "Leverkusen": "Bayer Leverkusen",
    "St Pauli": "FC St. Pauli",
    "Athletic Club": "Athletic Bilbao",
    "Nottingham": "Nottingham Forest",
    "Newcastle": "Newcastle United",
    "Leganés": "Leganes",
    "West Ham": "West Ham United",
    "Atlético Madrid": "Atletico Madrid",
    "Manchester Utd": "Manchester United",
    "Frankfurt": "Eintracht Frankfurt",
    "Wolfsburg": "VfL Wolfsburg",
    "Paris SG": "Paris Saint-Germain",
    "Tottenham": "Tottenham Hotspur",
    "Bochum": "VfL Bochum",
    "Heidenheim": "FC Heidenheim",
    "Stuttgart": "VfB Stuttgart",
    "Gladbach": "Borussia Monchengladbach"
}

df_players["club"] = df_players["club"].replace(clubes)


In [456]:
df_players

,player,position,club,league,Matches Played,Starts,Minutes
0,Max Aarons,DF,Bournemouth,Premier League,3,1,86
1,Max Aarons,DF,Valencia,La Liga,4,1,120
2,Rodrigo Abajas,DF,Valencia,La Liga,1,1,65
3,James Abankwah,DF,Udinese,Serie A,6,0,88
4,Keyliane Abdallah,FW,Marseille,Ligue 1,1,0,3
...,...,...,...,...,...,...,...
2849,Martin Ødegaard,MF,Arsenal,Premier League,30,26,2325
2850,Leo Østigård,DF,Hoffenheim,Bundesliga,11,11,927
2851,Leo Østigård,DF,Rennes,Ligue 1,16,14,1227
2852,Milan Đurić,FW,Monza,Serie A,18,13,1270


## 9. Comprobar coincidencias de clubes y jugadores

In [457]:
print("Clubes de players que no aparecen en injuries:")
print(
    df_players.loc[
        ~df_players["club"].isin(df_injuries["club"]),
        "club"
    ].unique()
)

print("\nJugadores que aparecen en players pero no en injuries:")
print(
    df_players.loc[
        ~df_players["player"].isin(df_injuries["player"]),
        "player"
    ].unique()
)


Clubes de players que no aparecen en injuries:
<ArrowStringArray>
[]
Length: 0, dtype: str

Jugadores que aparecen en players pero no en injuries:
<ArrowStringArray>
[    'Rodrigo Abajas',     'James Abankwah',  'Keyliane Abdallah',
   'Yunis Abdelhamid',      'Nabil Aberdin',  'Oliver Abildgaard',
     'Matthis Abline', 'Federico Accornero',  'Joshua Acheampong',
             'Adrián',
 ...
          'David Zec',     'Alessio Zerbin',     'Cedric Zesiger',
   'Michael Zetterer',        'Nathan Zeze',       'Melvin Zinga',
         'Luck Zogbé',    'Aristide Zossou',       'Edhy Zuliani',
        'Milan Đurić']
Length: 1110, dtype: str


Estas comprobaciones se mantienen porque son importantes para justificar que el `merge` no está uniendo datos de forma incorrecta.

## 10. Preparar los datos de lesiones

In [458]:
df_injuries["days"] = (
    df_injuries["days"]
    .str.replace(" days", "", regex=False)
    .astype(int)
)

print(df_injuries[["days", "games_missed"]].dtypes)


days            int64
games_missed    int64
dtype: object


In [459]:
df_ligas.columns

Index(['league', 'club', 'points', 'goals for', 'goals against'], dtype='str')

In [460]:
df_injuries.columns

Index(['season', 'injury', 'days', 'games_missed', 'player', 'position',
       'club', 'league'],
      dtype='str')

In [461]:
df_players.columns

Index(['player', 'position', 'club', 'league', 'Matches Played', 'Starts',
       'Minutes'],
      dtype='str')

In [462]:
df_ligas.columns

Index(['league', 'club', 'points', 'goals for', 'goals against'], dtype='str')

## 11. Próxima fase — SQL

El siguiente paso será llevar esta información a SQL.

La clasificación de las ligas (`df_ligas`) puede mantenerse como un DataFrame independiente para relacionarla posteriormente por club/temporada cuando se defina exactamente el modelo de datos.

In [463]:
df_players.shape
df_players['player'].nunique()
df_players['player'].duplicated().sum()

np.int64(151)

In [464]:
df_injuries.shape
df_injuries['player'].nunique()
df_injuries['player'].duplicated().sum()

np.int64(3739)

In [465]:
df_ligas['club'].nunique()

110

In [466]:
df_players['player'].duplicated().sum()

np.int64(151)

In [467]:
df_ligas[['league', 'club']].duplicated().sum()

np.int64(0)

Esta comprobación la hago para ver el tipo de dato nos devuelve la columna para convertirla en int64.

## 11. Agrupar lesiones por jugador y club

df_injuries_agg = (
    df_injuries
    .groupby(["player_name", "club"])[["Days", "Games missed"]]
    .sum()
    .reset_index()
)

print(df_injuries_agg.shape)
df_injuries_agg


## 12. Unir jugadores y lesiones

df_final = df_players.merge(
    df_injuries_agg,
    left_on=["Player", "Squad"],
    right_on=["player_name", "club"],
    how="left"
)

df_final[["Days", "Games missed"]] = (
    df_final[["Days", "Games missed"]].fillna(0)
)

df_final = df_final.drop(columns=["club", "player_name"])

print(df_final.shape)
df_final.head()


## 13. Comprobación final

Los valores `NaN` de lesiones se convierten en `0` porque en este contexto significan que el jugador no tiene un registro de lesión en el dataset utilizado.

Antes de pasar a SQL conviene revisar duplicados y tipos de datos.

print("Valores nulos:")
print(df_final.isna().sum())

print("\nFilas duplicadas:")
print(df_final.duplicated().sum())

print("\nTipos de datos:")
print(df_final.dtypes)


df_final

